In [2]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import open3d as o3d
from scipy.spatial.transform import Rotation as R 
import os 
import glob 
import pandas as pd 
from tqdm import tqdm
from scipy.ndimage import median_filter

In [3]:
color_intrinsics = {
    'width': 1280,
    'height': 720,
    'fx': 643.90087890625,
    'fy': 643.1365356445312,
    'cx': 650.2113037109375,
    'cy': 355.79559326171875,
    'model': "distortion.inverse_brown_conrady",
    'coeffs': [-0.05658450722694397, 0.06544225662946701,-0.0008694113348610699, 0.00016751799557823688,-0.020957745611667633]
}

depth_intrinsics = {
    'width': 1280,
    'height': 720,
    'fx': 650.0616455078125,
    'fy': 650.0616455078125,
    'cx': 649.5928955078125,
    'cy': 360.9415588378906,
    'model': "distortion.brown_conrady",
    'coeffs': [0.0, 0.0, 0.0, 0.0, 0.0]
}

ROI_X = 550
ROI_Y = 150
ROI_WIDTH = 300
ROI_HEIGHT = 330

roi = (ROI_X, ROI_Y, ROI_WIDTH, ROI_HEIGHT )


In [4]:
# 1. Đường dẫn đến thư mục chứa các file mask .npy
MASK_DIR = "masks_npy"

# =============================================================================
# HÀM CHÍNH (MAIN SCRIPT) - ĐÃ CẬP NHẬT LOGIC LOẠI BỎ
# =============================================================================

def get_masks_in_roi(roi):
    ROI_X, ROI_Y, ROI_W, ROI_H = roi
    results_list = []
    print(f"Bắt đầu quét thư mục mask: {MASK_DIR}")
    
    try:
        mask_files = [f for f in os.listdir(MASK_DIR) if f.endswith('.npy')]
    except FileNotFoundError:
        print(f"LỖI: Không tìm thấy thư mục: {MASK_DIR}")
        return
    
    if not mask_files:
        print(f"LỖI: Không tìm thấy file '.npy' nào.")
        return

    print(f"Tìm thấy {len(mask_files)} file mask. Bắt đầu tính toán...")

    skipped_masks = 0  # cần khai báo trước

    for filename in tqdm(mask_files, desc="Đang xử lý"):
        try:
            mask = np.load(os.path.join(MASK_DIR, filename))
            mask_uint8 = (mask * 255).astype(np.uint8)

            M = cv2.moments(mask_uint8)
            if M["m00"] > 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                
                # --- [MỚI] 4. Kiểm tra tâm có nằm trong mask không ---
                # Đảm bảo toạ độ không vượt quá kích thước ảnh
                h, w = mask_uint8.shape
                cx = min(max(cx, 0), w - 1)
                cy = min(max(cy, 0), h - 1)

                # Nếu pixel tại (cy, cx) là nền (giá trị 0) -> LOẠI BỎ
                if mask_uint8[cy, cx] == 0:
                    skipped_masks += 1
                    continue # Bỏ qua mask này, sang mask tiếp theo

                if not (ROI_X <= cx < ROI_X + ROI_W and ROI_Y <= cy < ROI_Y + ROI_H):
                    skipped_masks += 1
                    continue

                # Tìm bounding box của mask (full-image coords)
                coords = np.argwhere(mask_uint8 > 0)  # shape (N,2) mỗi dòng (cy, cx) full-image coords
                if coords.size == 0:
                    skipped_masks += 1
                    continue
                y_min, x_min = coords.min(axis=0)
                y_max, x_max = coords.max(axis=0) + 1

                # Crop mask theo bbox (crop coordinates)
                mask_cropped = mask_uint8[y_min:y_max, x_min:x_max]

                # --- NEW: mask pixels inside the CROP but expressed in FULL-IMAGE coordinates ---
                # coords_crop are local coordinates inside the cropped mask (row,col)
                coords_crop = np.argwhere(mask_cropped > 0)  # shape (M,2): (row_in_crop, col_in_crop)
                if coords_crop.size == 0:
                    skipped_masks += 1
                    continue
                # convert crop coords -> full-image coords by adding offsets (y_min, x_min)
                full_rows = coords_crop[:, 0] + int(y_min)
                full_cols = coords_crop[:, 1] + int(x_min)
                # pack as (cy, cx) full-image coords
                mask_pixels_in_crop_full = np.stack((full_rows, full_cols), axis=1)

                # For compatibility, also keep all mask pixels (full-image) if needed
                mask_pixels_full = coords  # already full-image (cy, cx)

                results_list.append({
                    'image_name': f"{filename.split('_')[0]}.png",
                    'mask_full': mask_uint8,
                    'mask': mask_cropped,
                    '2d_x': cx,
                    '2d_y': cy,
                    'x_min': int(x_min),
                    'y_min': int(y_min),
                    'x_max': int(x_max),
                    'y_max': int(y_max),
                    # mask_pixels: only pixels inside the crop but coordinates are full-image (row, col)
                    'mask_pixels': mask_pixels_in_crop_full,
                    # optional extra field if you need the full-mask pixels as well
                    'mask_pixels_full': mask_pixels_full
                })

        except Exception as e:
            print(f"Lỗi file {filename}: {e}")

    return results_list


In [5]:
def deproject_with_inverse_model(u_distorted, v_distorted, depth_meters, intrinsics):
    fx = intrinsics['fx']; fy = intrinsics['fy']; cx = intrinsics['cx']; cy = intrinsics['cy']
    k1, k2, p1, p2, k5 = intrinsics['coeffs']
    x_distorted_normalized = (u_distorted - cx) / fx
    y_distorted_normalized = (v_distorted - cy) / fy
    r2 = x_distorted_normalized**2 + y_distorted_normalized**2; r4 = r2**2; r6 = r2**3
    radial_distortion = 1 + k1*r2 + k2*r4 + k5*r6
    tangential_distortion_x = 2*p1*x_distorted_normalized*y_distorted_normalized + p2*(r2 + 2*x_distorted_normalized**2)
    tangential_distortion_y = p1*(r2 + 2*y_distorted_normalized**2) + 2*p2*x_distorted_normalized*y_distorted_normalized
    x_undistorted_normalized = x_distorted_normalized * radial_distortion + tangential_distortion_x
    y_undistorted_normalized = y_distorted_normalized * radial_distortion + tangential_distortion_y
    z_c = depth_meters; x_c = x_undistorted_normalized * z_c; y_c = y_undistorted_normalized * z_c
    return x_c, y_c, z_c

In [6]:
def fit_plane_3d_ransac(points_3d, threshold=0.01, max_trials=100, use_farthest_pair=True):
    """
    Fit mặt phẳng 3D tổng quát bằng RANSAC.
    Plane equation: ax + by + cz + d = 0

    If use_farthest_pair is True, this routine finds the two farthest-apart
    points (global) and in each RANSAC trial uses them together with a third
    randomly sampled point to form the candidate plane. This can improve
    stability when points form a long, planar patch.
    """
    if len(points_3d) < 3:
        return None

    N = len(points_3d)
    best_inliers = None
    best_plane = None
    max_inliers = 0

    # Precompute farthest pair if requested (may be O(N^2))
    far_i, far_j = None, None
    if use_farthest_pair and N <= 3000:  # guard against very large N
        # compute pairwise squared distances efficiently
        diff = points_3d[:, None, :] - points_3d[None, :, :]
        dist2 = np.sum(diff * diff, axis=2)
        # ignore diagonal
        np.fill_diagonal(dist2, -1.0)
        idx = np.argmax(dist2)
        far_i = idx // N
        far_j = idx % N
    else:
        # fallback: pick two random distinct points
        rand_idx = np.random.choice(N, 2, replace=False)
        far_i, far_j = int(rand_idx[0]), int(rand_idx[1])

    np.random.seed(42)

    for _ in range(max_trials):
        try:
            # pick third point (not equal to far_i/far_j)
            if use_farthest_pair:
                # ensure third index differs
                choices = list(range(N))
                choices.remove(far_i)
                choices.remove(far_j)
                if len(choices) == 0:
                    continue
                k = np.random.choice(choices)
                p1 = points_3d[far_i]
                p2 = points_3d[far_j]
                p3 = points_3d[k]
            else:
                idx = np.random.choice(N, 3, replace=False)
                p1, p2, p3 = points_3d[idx]

            v1 = p2 - p1
            v2 = p3 - p1
            normal = np.cross(v1, v2)
            norm_length = np.linalg.norm(normal)
            if norm_length < 1e-6:
                continue
            normal = normal / norm_length
            d = -np.dot(normal, p1)

            # distances from all points to plane (since normal is normalized, denom=1)
            distances = np.abs(points_3d @ normal + d)
            inlier_mask = distances < threshold
            num_inliers = np.sum(inlier_mask)

            if num_inliers > max_inliers:
                max_inliers = int(num_inliers)
                best_inliers = inlier_mask
                best_plane = (normal, d)
        except Exception:
            # catch any numerical issue and continue
            continue

    if best_plane is None or max_inliers < 3:
        return None

    inlier_points = points_3d[best_inliers]
    centroid = np.mean(inlier_points, axis=0)
    centered = inlier_points - centroid
    _, _, vh = np.linalg.svd(centered)
    refined_normal = vh[2, :]
    # ensure normal is normalized
    refined_normal = refined_normal / (np.linalg.norm(refined_normal) + 1e-12)
    refined_d = -np.dot(refined_normal, centroid)

    # recompute inlier mask using refined plane
    distances_refined = np.abs(points_3d @ refined_normal + refined_d)
    refined_inlier_mask = distances_refined < threshold
    refined_num_inliers = int(np.sum(refined_inlier_mask))

    return {
        'normal': refined_normal,
        'd': refined_d,
        'inlier_mask': refined_inlier_mask,
        'num_inliers': refined_num_inliers,
        'centroid': centroid
    }

In [7]:
def preprocess_depth_image(depth_image_raw, scale=1000.0, fill_holes=True):
    """
    Chuyển đổi ảnh depth sang mét và lấp đầy các lỗ trống (holes).
    """
    Z = depth_image_raw.astype(np.float32) / scale
    valid = (Z > 0.01) & (Z < 5.0) & np.isfinite(Z)
    
    if fill_holes:
        Z_filled = median_filter(Z, size=3)
        fill_mask = (~valid) & (cv2.dilate(valid.astype(np.uint8), np.ones((3,3), np.uint8)) > 0)
        Z[fill_mask] = Z_filled[fill_mask]
    
    return Z

In [8]:
MIN_VALID_METERS = 0.01
MAX_VALID_METERS = 5.0
HOLE_FILL_BUCKETS = {
    "SMALL": {
        'max_dim_thresh': 12,
        'median_size': 18,
        'dilate_kernel_size': 12
    },
    "MEDIUM": {
        'max_dim_thresh': 45,
        'median_size': 55,
        'dilate_kernel_size': 45 
    },
    "LARGE": {
        'max_dim_thresh': 300,
        'median_size': 85,
        'dilate_kernel_size': 75,
    }
}

MIN_VALID_METERS = 0.01
MAX_VALID_METERS = 5.0

def adaptive_fill_holes(depth_meters_raw, config_buckets):
    """
    Lấp lỗ thích ứng dựa trên kích thước lỗ (buckets) nhưng không dùng object mask.
    - depth_meters_raw: 2D depth map (float, meters)
    - config_buckets: dict, mỗi bucket có median_size, max_dim_thresh, dilate_kernel_size
    """
    Z = depth_meters_raw.astype(np.float32) / 1000.0
    valid = (Z > MIN_VALID_METERS) & (Z < MAX_VALID_METERS) & np.isfinite(Z)
    
    # Tạo mask lỗ
    holes_mask = (~valid).astype(np.uint8)
    
    # Connected components
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(holes_mask, 8, cv2.CV_32S)
    if num_labels <= 1:
        return Z
    
    # Median filter cho từng bucket (chỉ 1 lần mỗi size)
    filled_images = {}
    for name, params in config_buckets.items():
        size = params['median_size']
        if size not in filled_images:
            filled_images[size] = median_filter(Z, size=size)
    
    for i in range(1, num_labels):
        stat = stats[i]
        width, height = stat[cv2.CC_STAT_WIDTH], stat[cv2.CC_STAT_HEIGHT]
        max_dim = max(width, height)
        
        # Chọn bucket
        chosen_bucket = None
        for name in ["SMALL", "MEDIUM", "LARGE"]:
            if name not in config_buckets:
                continue
            params = config_buckets[name]
            if max_dim <= params['max_dim_thresh']:
                chosen_bucket = params
                break
        if chosen_bucket is None:
            continue  # lỗ quá lớn
        
        # Dilate valid region
        dilate_kernel = np.ones((chosen_bucket['dilate_kernel_size'],) * 2, np.uint8)
        dilated_valid = cv2.dilate(valid.astype(np.uint8), dilate_kernel) > 0
        
        # Tạo mask fill
        blob_mask = (labels == i)
        fill_mask = blob_mask & dilated_valid
        
        # Lấp
        Z[fill_mask] = filled_images[chosen_bucket['median_size']][fill_mask]
    
    return Z


In [ ]:
Z_THRESHOLD_METERS = 0.005 
ROBOT_COORD_3D = np.array([0.6524, -0.2263, 0.9340])
OUTPUT_CSV = "Submission_3D.csv"

def process_all_masks(roi, depth_img_dir, depth_intr, colors_intr, 
                      neighbor_radius=15, plane_threshold=0.005, 
                      inlier_threshold=0.008, distance_threshold=0.01, sigma_ratio=0.5):
    """
    Pipeline cập nhật:
      - Lấy mask và tâm 2D (với bbox offsets) từ `get_masks_in_roi` (mask_pixels là pixels trong crop nhưng full-image coords).
      - Dùng vòng tròn pixel (neighbor_radius) trên ảnh depth để thu thập điểm 3D quanh tâm -> fit plane bằng RANSAC.
      - Deproject tất cả pixel trong crop (mask_pixels) sang 3D dùng depth image + depth_intr.
      - Lọc các điểm mask-3D này theo khoảng cách đến plane (distance_threshold).
      - Tính centroid mới = weighted mean các điểm còn lại (trọng số theo khoảng cách pixel tới tâm).

    Args:
        roi: tuple (x, y, width, height)
        depth_img_dir: đường dẫn folder chứa depth images
        depth_intr: dict {'fx','fy','cx','cy','coeffs'}
        colors_intr: dict {'fx','fy','cx','cy'}
        neighbor_radius: bán kính vùng lân cận để fit plane (pixels)
        plane_threshold: ngưỡng RANSAC (meters)
        inlier_threshold: ngưỡng lọc inlier (meters)
        distance_threshold: ngưỡng để coi điểm thuộc plane (meters)
        sigma_ratio: sigma = neighbor_radius * sigma_ratio used for weight computation
    Returns:
        results_df: DataFrame với columns [image_name, 2d_x, 2d_y, x_3d, y_3d, z_3d]
    """
    print("=" * 60)
    print("BƯỚC 1: Lấy masks trong ROI")
    print("=" * 60)
    masks_data = get_masks_in_roi(roi)
    if masks_data is None or len(masks_data) == 0:
        print("Không có mask nào trong ROI!")
        return None
    print(f"✓ Tìm thấy {len(masks_data)} masks hợp lệ\n")
    print("=" * 60)
    print("BƯỚC 2: Nhóm masks theo image")
    print("=" * 60)
    masks_by_image = {}
    for mask_data in masks_data:
        img_name = mask_data['image_name']
        if img_name not in masks_by_image:
            masks_by_image[img_name] = []
        masks_by_image[img_name].append(mask_data)
    print(f"✓ Có {len(masks_by_image)} images khác nhau\n")
    print("=" * 60)
    print("BƯỚC 3: Xử lý từng image")
    print("=" * 60)
    results = []
    failed_count = 0
    total_masks = 0
    depth_meters_cache = {} 
    for img_name, masks_list in tqdm(masks_by_image.items(), desc="Processing images"):
        depth_path = os.path.join(depth_img_dir, img_name)
        if not os.path.exists(depth_path):
            print(f"\n⚠ Không tìm thấy: {depth_path}")
            failed_count += len(masks_list)
            total_masks += len(masks_list)
            continue
        depth_img_raw = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
        if depth_img_raw is None:
            print(f"\n⚠ Không load được: {depth_path}")
            failed_count += len(masks_list)
            total_masks += len(masks_list)
            continue
        # Fill hole
        #depth_meters_cache[img_name] = preprocess_depth_image(depth_img_raw, 1000.0, fill_holes=True)
        depth_meters_cache[img_name] = adaptive_fill_holes(depth_img_raw, HOLE_FILL_BUCKETS)
        depth_meters = depth_meters_cache[img_name]

        for mask_data in masks_list:
            total_masks += 1
            try:
                center_u = int(round(mask_data['2d_x']))
                center_v = int(round(mask_data['2d_y']))

                # --- Bước A: Lấy điểm 3D trong vòng tròn pixel (neighbor_radius) từ ảnh depth ---
                h, w = depth_meters.shape
                u_min = max(0, center_u - neighbor_radius)
                u_max = min(w, center_u + neighbor_radius + 1)
                v_min = max(0, center_v - neighbor_radius)
                v_max = min(h, center_v + neighbor_radius + 1)

                uu, vv = np.meshgrid(np.arange(u_min, u_max), np.arange(v_min, v_max), indexing='xy')
                uu = uu.reshape(-1)
                vv = vv.reshape(-1)
                # circle mask
                d2 = (uu - center_u)**2 + (vv - center_v)**2
                circle_mask = d2 <= (neighbor_radius**2)
                uu = uu[circle_mask]
                vv = vv[circle_mask]

                if len(uu) == 0:
                    failed_count += 1
                    continue

                neigh_points = []
                for (u_px, v_px) in zip(uu, vv):
                    # read depth at (v_px, u_px) (row, col) -> meters
                    if v_px < 0 or v_px >= depth_meters.shape[0] or u_px < 0 or u_px >= depth_meters.shape[1]:
                        continue
                    z_val = depth_meters[v_px, u_px]
                    if z_val <= 0 or not np.isfinite(z_val):
                        continue
                    # Deproject using colors intrinsic
                    x_d, y_d, z_d = deproject_with_inverse_model(u_px, v_px, z_val, colors_intr)
                    neigh_points.append(np.array([x_d, y_d, z_d]))

                if len(neigh_points) < 3:
                    failed_count += 1
                    continue

                neigh_pts = np.array(neigh_points)

                # --- Bước B: Fit plane trên neighborhood points ---
                plane_result = fit_plane_3d_ransac(neigh_pts, threshold=plane_threshold)
                if plane_result is None:
                    failed_count += 1
                    continue

                normal = plane_result['normal']
                d_plane = plane_result['d']

                # --- Bước C: Deproject mask pixels (pixels inside crop) sang 3D directly ---
                mask_pixels = mask_data.get('mask_pixels', None)  # shape (M,2) of (row, col) in full-image coords
                if mask_pixels is None or mask_pixels.size == 0:
                    failed_count += 1
                    continue

                mask_pts3d = []
                mask_pts_px_dist = []  # pixel distance to center (for weighting)
                for (row_full, col_full) in mask_pixels:
                    r = int(row_full); c = int(col_full)
                    if r < 0 or r >= depth_meters.shape[0] or c < 0 or c >= depth_meters.shape[1]:
                        continue
                    z_val = depth_meters[r, c]
                    if z_val <= 0 or not np.isfinite(z_val):
                        continue
                    x3, y3, z3 = deproject_with_inverse_model(c, r, z_val, colors_intr)
                    mask_pts3d.append([x3, y3, z3])
                    # pixel-space distance to centroid (use euclidean on pixels)
                    dist_px = np.sqrt((c - center_u)**2 + (r - center_v)**2)
                    mask_pts_px_dist.append(dist_px)

                if len(mask_pts3d) == 0:
                    failed_count += 1
                    continue

                mask_pts3d = np.array(mask_pts3d)
                mask_pts_px_dist = np.array(mask_pts_px_dist)

                # --- Bước D: Lọc mask points theo khoảng cách đến plane ---
                dists = np.abs(mask_pts3d @ normal + d_plane)
                near_mask = dists <= distance_threshold
                if not np.any(near_mask):
                    failed_count += 1
                    continue

                selected_pts = mask_pts3d[near_mask]
                selected_dists_px = mask_pts_px_dist[near_mask]

                # Compute weighted centroid of selected points (weights based on pixel distance)
                sigma = max(1e-6, neighbor_radius * sigma_ratio)
                weights = np.exp(-0.5 * (selected_dists_px / sigma)**2)
                weight_total = np.sum(weights)
                if weight_total == 0:
                    failed_count += 1
                    continue
                weighted_sum = np.sum(selected_pts * weights[:, None], axis=0)
                new_centroid = weighted_sum / weight_total

                results.append({
                    'image_filename': mask_data['image_name'],
                    '2d_x': mask_data['2d_x'],
                    '2d_y': mask_data['2d_y'],
                    'x': float(new_centroid[0]),
                    'y': float(new_centroid[1]),
                    'z': float(new_centroid[2])
                })

            except Exception as e:
                print(f"\nLỗi mask trong {img_name}: {e}")
                failed_count += 1
                continue

    df_all = pd.DataFrame(results)
    if df_all.empty:
        return
    print("Đang lọc kết quả cuối cùng...")
    df_all['min_z'] = df_all.groupby('image_filename')['z'].transform('min')
    cands = df_all[df_all['z'] <= (df_all['min_z'] + Z_THRESHOLD_METERS)].copy()
    cands['dist'] = np.linalg.norm(cands[['x','y','z']].values - ROBOT_COORD_3D, axis=1)
    final_df = cands.sort_values(['dist'], ascending=False).drop_duplicates('image_filename')
    final_df['image_filename'] = 'image_' + final_df['image_filename'].str.replace('image_', '', regex=False)
    final_df[['image_filename', 'x', 'y', 'z']].to_csv(OUTPUT_CSV, index=False, float_format='%.3f')
    print(f"Hoàn tất! Đã lưu vào {OUTPUT_CSV}")


In [ ]:
DEPTH_IMAGE_DIR = "train/depth"
process_all_masks(
        roi=roi,
        depth_img_dir=DEPTH_IMAGE_DIR,
        depth_intr=depth_intrinsics,
        colors_intr=color_intrinsics,
        neighbor_radius=5,        # Bán kính fit plane
        plane_threshold=0.01,     # 5mm RANSAC
        distance_threshold=0.01    # thuộc plane
    )
    

BƯỚC 1: Lấy masks trong ROI
Bắt đầu quét thư mục mask: masks_npy
Tìm thấy 1248 file mask. Bắt đầu tính toán...


Đang xử lý: 100%|██████████| 1248/1248 [00:29<00:00, 41.68it/s]


✓ Tìm thấy 694 masks hợp lệ

BƯỚC 2: Nhóm masks theo image
✓ Có 369 images khác nhau

BƯỚC 3: Xử lý từng image


Processing images:   3%|▎         | 11/369 [18:15<9:47:41, 98.50s/it] 

In [20]:
import pandas as pd
import numpy as np
import os

# --- 1. Định nghĩa Tên tệp và Hằng số ---
PREDICT_CSV = "Submission_3D.csv"
GT_CSV = "public_train.csv"

# Tên file output mới để lưu kết quả so sánh
OUTPUT_CSV_PATH = "comparison_metrics_normalized.csv"

# [THÊM MỚI] Ngưỡng sai số hợp lệ (0.05 mét = 5cm)
VALIDITY_THRESHOLD = 0.05

# --- 2. Hàm tính toán ---
def calculate_metrics():
    """
    Tải file dự đoán và file ground truth, so sánh,
    tính MCE chuẩn hóa, lưu kết quả và in ra các chỉ số trung bình.
    """
    
    # --- 3. Tải dữ liệu ---
    try:
        cols_to_use = ['image_filename', 'x', 'y', 'z']
        df_predict = pd.read_csv(PREDICT_CSV, usecols=cols_to_use)
        df_gt = pd.read_csv(GT_CSV, usecols=cols_to_use)
        
    except FileNotFoundError as e:
        print(f"LỖI: Không tìm thấy tệp: {e.filename}")
        return
    except ValueError as e:
        print(f"LỖI: Tệp CSV có thể thiếu cột. {e}")
        print(f"Hãy đảm bảo cả hai tệp đều có cột: {cols_to_use}")
        return
    except Exception as e:
        print(f"LỖI khi đọc CSV: {e}")
        return

    print(f"Đã tải {len(df_predict)} dự đoán từ: {os.path.basename(PREDICT_CSV)}")
    print(f"Đã tải {len(df_gt)} ground truth từ: {os.path.basename(GT_CSV)}")

    # --- 4. Gộp (Merge) hai DataFrame ---
    df_merged = pd.merge(
        df_predict, 
        df_gt, 
        on='image_filename', 
        suffixes=('_pred', '_gt')
    )
    
    if df_merged.empty:
        print("LỖI: Không có image_filename nào trùng khớp giữa hai tệp.")
        return

    print(f"Tìm thấy {len(df_merged)} ảnh trùng khớp để so sánh.")

    # --- 5. Tính toán Sai số (Error) ---
    
    # 5a. Tính sai số (deviation) cho từng trục (giữ nguyên)
    df_merged['x_error'] = df_merged['x_pred'] - df_merged['x_gt']
    df_merged['y_error'] = df_merged['y_pred'] - df_merged['y_gt']
    df_merged['z_error'] = df_merged['z_pred'] - df_merged['z_gt']

    # 5b. Tính khoảng cách Euclidean 3D thô (để làm cơ sở)
    pred_coords = df_merged[['x_pred', 'y_pred', 'z_pred']].to_numpy()
    gt_coords = df_merged[['x_gt', 'y_gt', 'z_gt']].to_numpy()
    df_merged['mce_distance'] = np.linalg.norm(pred_coords - gt_coords, axis=1)

    # 5c. [LOGIC MỚI] Tính MCE chuẩn hóa và giới hạn
    # Chia sai số thô cho ngưỡng, sau đó giới hạn giá trị tối đa là 1.0
    df_merged['normalized_mce'] = (df_merged['mce_distance'] / VALIDITY_THRESHOLD).clip(upper=1.0)


    # --- 6. Lưu file CSV kết quả chi tiết ---
    
    # [SỬA] Thêm cột mới vào file output
    output_columns = [
        'image_filename', 
        'x_pred', 'x_gt', 'x_error',
        'y_pred', 'y_gt', 'y_error',
        'z_pred', 'z_gt', 'z_error',
        'mce_distance',     # Sai số thô (mét)
        'normalized_mce'    # Sai số đã chuẩn hóa và giới hạn
    ]
    
    df_results = df_merged[output_columns]
    
    try:
        df_results.to_csv(OUTPUT_CSV_PATH, index=False, float_format='%.3f')
        print(f"\nĐã lưu kết quả so sánh chi tiết vào: {OUTPUT_CSV_PATH}")
    except Exception as e:
        print(f"\nLỖI: Không thể lưu file CSV: {e}")

    # --- 7. Tính toán và In ra các chỉ số trung bình đã cập nhật ---
    
    # [SỬA] Chỉ số chính là MCE chuẩn hóa trung bình
    mean_normalized_mce = df_merged['normalized_mce'].mean()
    
    # [THÊM MỚI] Tính tỷ lệ hợp lệ
    accuracy_rate = (df_merged['mce_distance'] < VALIDITY_THRESHOLD).mean() * 100
    
    # Giữ lại các chỉ số thô để tham khảo
    mean_raw_mce = df_merged['mce_distance'].mean()
    mean_x_deviation = df_merged['x_error'].mean()
    mean_y_deviation = df_merged['y_error'].mean()
    mean_z_deviation = df_merged['z_error'].mean()
    
    print("\n" + "="*50)
    print("--- 🏆 CHỈ SỐ CHUẨN HÓA (Normalized MCE) ---")
    print(f"Điểm MCE chuẩn hóa trung bình: {mean_normalized_mce:.3f} (0 = hoàn hảo, 1 = sai số lớn)")
    print(f"Tỷ lệ hợp lệ (sai số < {VALIDITY_THRESHOLD*1000:.0f}mm): {accuracy_rate:.2f}%")
    print("="*50)
    
    print("\n--- 📊 CHỈ SỐ THÔ (Để tham khảo) ---")
    print(f"MCE trung bình (Khoảng cách 3D thô): {mean_raw_mce:.3f} mét")
    print(f"Độ lệch X trung bình (pred - gt): {mean_x_deviation:.3f} mét")
    print(f"Độ lệch Y trung bình (pred - gt): {mean_y_deviation:.3f} mét")
    print(f"Độ lệch Z trung bình (pred - gt): {mean_z_deviation:.3f} mét")


# --- Chạy hàm chính ---
if __name__ == "__main__":
    calculate_metrics()

Đã tải 369 dự đoán từ: Submission_3D.csv
Đã tải 350 ground truth từ: public_train.csv
Tìm thấy 349 ảnh trùng khớp để so sánh.

Đã lưu kết quả so sánh chi tiết vào: comparison_metrics_normalized.csv

--- 🏆 CHỈ SỐ CHUẨN HÓA (Normalized MCE) ---
Điểm MCE chuẩn hóa trung bình: 0.185 (0 = hoàn hảo, 1 = sai số lớn)
Tỷ lệ hợp lệ (sai số < 50mm): 97.13%

--- 📊 CHỈ SỐ THÔ (Để tham khảo) ---
MCE trung bình (Khoảng cách 3D thô): 0.013 mét
Độ lệch X trung bình (pred - gt): -0.003 mét
Độ lệch Y trung bình (pred - gt): 0.000 mét
Độ lệch Z trung bình (pred - gt): -0.001 mét


In [32]:
def visualize_submission_plane(image_filename=None, pred_csv='Submission_3D.csv', gt_csv='public_train.csv', depth_dir='train/depth', neighbor_radius=20, plane_threshold=0.01, depth_intr=depth_intrinsics, color_intr=color_intrinsics):
    """
    Visualize predicted centroid and ground-truth for a single image, with the point cloud of
    the fitted plane (inliers) and their orthogonal projection onto the plane.

    If image_filename is None, picks the first entry in `pred_csv`.
    """
    import os
    import numpy as np
    import pandas as pd
    import open3d as o3d
    import cv2

    # Load predictions
    if not os.path.exists(pred_csv):
        print(f"Prediction CSV not found: {pred_csv}")
        return
    preds = pd.read_csv(pred_csv)
    if preds.empty:
        print("No predictions in CSV")
        return

    if image_filename is None:
        image_filename = preds.iloc[0]['image_filename']
        print(f"Using first prediction: {image_filename}")

    pred_row = preds[preds['image_filename'] == image_filename]
    if pred_row.empty:
        print(f"No prediction for {image_filename}")
        return
    pred_row = pred_row.iloc[0]
    pred_3d = np.array([pred_row['x'], pred_row['y'], pred_row['z']], dtype=float)

    # Load ground truth if available
    gt_3d = None
    if os.path.exists(gt_csv):
        try:
            gt_df = pd.read_csv(gt_csv)
            gt_row = gt_df[gt_df['image_filename'] == image_filename]
            if len(gt_row) > 0:
                gt_row = gt_row.iloc[0]
                gt_3d = np.array([gt_row['x'], gt_row['y'], gt_row['z']], dtype=float)
        except Exception as e:
            print('Could not read GT CSV:', e)

    # Load depth
    depth_path = os.path.join(depth_dir, image_filename.split('_')[1])
    if not os.path.exists(depth_path):
        print(f"Depth file not found: {depth_path}")
        return
    depth_raw = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
    if depth_raw is None:
        print('Failed to load depth image')
        return
    depth_m = depth_raw.astype(np.float32)
    if depth_raw.max() > 100:  # likely mm
        depth_m = depth_raw.astype(np.float32) / 1000.0

    H, W = depth_m.shape

    # Project predicted 3D to image plane to choose neighborhood center
    X, Y, Z = pred_3d
    if Z == 0 or not np.isfinite(Z):
        print('Predicted Z invalid, cannot project to image plane')
        return
    u_center = int(round((X * color_intr['fx'] / Z) + color_intr['cx']))
    v_center = int(round((Y * color_intr['fy'] / Z) + color_intr['cy']))

    # Gather neighborhood pixels in circle
    u_min = max(0, u_center - neighbor_radius)
    u_max = min(W, u_center + neighbor_radius + 1)
    v_min = max(0, v_center - neighbor_radius)
    v_max = min(H, v_center + neighbor_radius + 1)

    uu, vv = np.meshgrid(np.arange(u_min, u_max), np.arange(v_min, v_max), indexing='xy')
    uu = uu.reshape(-1); vv = vv.reshape(-1)
    d2 = (uu - u_center)**2 + (vv - v_center)**2
    circle_mask = d2 <= (neighbor_radius**2)
    uu = uu[circle_mask]; vv = vv[circle_mask]

    pts3d = []
    pts_uv = []
    for u_px, v_px in zip(uu, vv):
        if v_px < 0 or v_px >= H or u_px < 0 or u_px >= W:
            continue
        z = depth_m[v_px, u_px]
        if z <= 0 or not np.isfinite(z):
            continue
        # deproject using depth intrinsics
        x, y, z = deproject_with_inverse_model(u_px, v_px, z, depth_intr)
        pts3d.append([x, y, z])
        pts_uv.append([u_px, v_px])

    if len(pts3d) == 0:
        print('No valid depth points in neighborhood')
        return
    pts3d = np.array(pts3d)

    # Fit plane
    plane = fit_plane_3d_ransac(pts3d, threshold=plane_threshold, max_trials=200)
    if plane is None:
        print('Plane fit failed')
        # still visualize raw points and pred/gt
        p_all = o3d.geometry.PointCloud(); p_all.points = o3d.utility.Vector3dVector(pts3d); p_all.paint_uniform_color([0.7,0.7,0.7])
        geoms = [p_all]
        if gt_3d is not None:
            s = o3d.geometry.TriangleMesh.create_sphere(radius=0.01); s.translate(gt_3d); s.paint_uniform_color([0.2,1.0,0.2]); geoms.append(s)
        s2 = o3d.geometry.TriangleMesh.create_sphere(radius=0.01); s2.translate(pred_3d); s2.paint_uniform_color([1.0,0.2,0.2]); geoms.append(s2)
        geoms.append(o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.05))
        o3d.visualization.draw_geometries(geoms, window_name=f'Viz {image_filename}')
        return

    normal = plane['normal']
    d_plane = plane['d']
    inlier_mask = plane['inlier_mask']
    inliers = pts3d[inlier_mask]

    # Project inliers orthogonally onto plane
    # For point p: p_proj = p - (p.dot(n) + d) * n
    signed = (inliers @ normal) + d_plane
    projected = inliers - np.outer(signed, normal)

    # Create plane mesh for visualization
    plane_center = plane['centroid']
    # build two orthonormal axes on plane
    n = normal / np.linalg.norm(normal)
    # choose arbitrary vector not parallel to n
    arbitrary = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(arbitrary, n)) > 0.9:
        arbitrary = np.array([0.0, 1.0, 0.0])
    axis_u = np.cross(n, arbitrary)
    axis_u = axis_u / np.linalg.norm(axis_u)
    axis_v = np.cross(n, axis_u)
    axis_v = axis_v / np.linalg.norm(axis_v)

    # size of plane patch: based on neighbor_radius and approximate focal depth
    avg_depth = np.mean(inliers[:,2])
    # approximate pixel to meter scale at that depth: 1 pixel ~ 1/fx * z meters
    meter_per_pixel = avg_depth / depth_intr['fx']
    patch_size_m = neighbor_radius * meter_per_pixel * 1.6

    # create grid
    s = patch_size_m
    res = 2
    us = np.linspace(-s, s, 2*res+1)
    vs = np.linspace(-s, s, 2*res+1)
    verts = []
    for uu_ in us:
        for vv_ in vs:
            pt = plane_center + uu_ * axis_u + vv_ * axis_v
            verts.append(pt)
    verts = np.array(verts)
    # create mesh triangles
    mesh = o3d.geometry.TriangleMesh()
    mesh.vertices = o3d.utility.Vector3dVector(verts)
    triangles = []
    nx = len(us)
    ny = len(vs)
    for i in range(nx-1):
        for j in range(ny-1):
            idx0 = i*ny + j
            idx1 = (i+1)*ny + j
            idx2 = (i+1)*ny + (j+1)
            idx3 = i*ny + (j+1)
            triangles.append([idx0, idx1, idx2])
            triangles.append([idx0, idx2, idx3])
    mesh.triangles = o3d.utility.Vector3iVector(np.array(triangles))
    mesh.compute_vertex_normals()
    mesh.paint_uniform_color([0.9,0.9,0.6])
    mesh.compute_triangle_normals()
    mesh.paint_uniform_color([0.9,0.9,0.6])
    mesh.translate([0,0,0])
    mesh = mesh

    # Build Open3D point clouds and markers
    p_all = o3d.geometry.PointCloud(); p_all.points = o3d.utility.Vector3dVector(pts3d); p_all.paint_uniform_color([0.7,0.7,0.7])
    p_in = o3d.geometry.PointCloud(); p_in.points = o3d.utility.Vector3dVector(inliers); p_in.paint_uniform_color([0.1,0.3,0.8])
    p_proj = o3d.geometry.PointCloud(); p_proj.points = o3d.utility.Vector3dVector(projected); p_proj.paint_uniform_color([0.1,0.8,0.6])

    geoms = [p_all, p_in, p_proj, mesh]

    # predicted centroid (red)
    s_pred = o3d.geometry.TriangleMesh.create_sphere(radius=0.01); s_pred.translate(pred_3d); s_pred.paint_uniform_color([1.0,0.2,0.2]); s_pred.compute_vertex_normals(); geoms.append(s_pred)
    # ground truth centroid (green)
    if gt_3d is not None:
        s_gt = o3d.geometry.TriangleMesh.create_sphere(radius=0.01); s_gt.translate(gt_3d); s_gt.paint_uniform_color([0.2,1.0,0.2]); s_gt.compute_vertex_normals(); geoms.append(s_gt)

    geoms.append(o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.05))

    print(f'Points total: {len(pts3d)}, inliers: {len(inliers)}, projected: {len(projected)}')
    print('Predicted 3D:', pred_3d)
    if gt_3d is not None:
        print('GT 3D:', gt_3d)

    o3d.visualization.draw_geometries(geoms, window_name=f'Plane viz {image_filename}')

# Example: call with the first entry in Submission_3D.csv if run as script
if __name__ == '__main__':
    visualize_submission_plane(image_filename='image_0265.png',neighbor_radius=12, plane_threshold=0.008)


Points total: 440, inliers: 440, projected: 440
Predicted 3D: [ 0.142 -0.171  1.081]
GT 3D: [ 0.14  -0.184  1.085]
